# Equality Classifier — Análisis de Resultados

**Archivos de entrada:**
- `primary.csv` — clasificación primaria (392 k textos): `id, label, conf`
- `crossval.csv` — validación cruzada 3 modelos, 1 500 textos: `id, label_gemini, conf_gemini, dimension, country, label_groq, conf_groq, label_or, conf_or`
- `Manual_Classification_Labeled.csv` — 750 textos etiquetados por humano: `ID, Dimension, Country, Text, Label Primary, Label Manual, Notes`

**Etiquetas:** `PRO_EQUALITY` | `ANTI_EQUALITY` | `NEUTRAL`  
**Dimensiones:** education · gender · health · income · wealth  
**Modelos crossval:** Gemini Flash 2.0 Lite · Groq Llama-3.1-8B · OpenRouter Llama-3.3-70B


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, cohen_kappa_score
)
from collections import Counter
import os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130})

CATS    = ['PRO_EQUALITY', 'ANTI_EQUALITY', 'NEUTRAL']
COLORS  = {'PRO_EQUALITY': '#2196F3', 'ANTI_EQUALITY': '#F44336', 'NEUTRAL': '#9E9E9E'}
DIMS    = ['education', 'gender', 'health', 'income', 'wealth']
MODELS  = {'Gemini': 'label_gemini', 'Groq': 'label_groq', 'OpenRouter': 'label_or'}

RESULTS_DIR = 'Results/'
ORIG_PATH   = '/content/drive/MyDrive/RA/RA/NOW/DataProcesing/LLMs/2. Dimensions/Results/Treashold/AllDimensions.csv'


## 1. Carga de datos

In [ ]:
primary  = pd.read_csv(RESULTS_DIR + 'primary.csv')
crossval = pd.read_csv(RESULTS_DIR + 'crossval.csv')
manual   = pd.read_csv(RESULTS_DIR + 'Manual_Classification_Labeled.csv')

# Normalizar columnas manual
manual.columns = [c.strip() for c in manual.columns]
manual = manual.rename(columns={
    'ID': 'id', 'Dimension': 'dimension', 'Country': 'country',
    'Text': 'text', 'Label Primary': 'label_primary',
    'Label Manual': 'label_manual', 'Notes': 'notes'
})
manual[['label_primary', 'label_manual']] = manual[['label_primary', 'label_manual']].apply(
    lambda c: c.str.strip())

# Voto mayoritario en crossval
def majority(row):
    votes = [row['label_gemini'], row['label_groq'], row['label_or']]
    top, n = Counter(votes).most_common(1)[0]
    return pd.Series({'majority': top, 'consensus': n / 3})

crossval[['majority', 'consensus']] = crossval.apply(majority, axis=1)

print(f'primary  : {len(primary):,} filas  |  cols: {primary.columns.tolist()}')
print(f'crossval : {len(crossval):,} filas  |  cols: {crossval.columns.tolist()}')
print(f'manual   : {len(manual):,} filas   |  cols: {manual.columns.tolist()}')


## 2. Clasificación primaria — visión global

In [ ]:
dist = primary['label'].value_counts().reindex(CATS, fill_value=0)
pct  = (dist / len(primary) * 100).round(1)

summary = pd.DataFrame({'n': dist, '%': pct})
summary.loc['TOTAL'] = [len(primary), 100.0]
display(summary)

print(f"\nConfianza media            : {primary['conf'].mean():.3f}")
print(f"Conf < 0.6 (baja confianza): {(primary['conf'] < 0.6).sum():,} ({(primary['conf'] < 0.6).mean():.1%})")
print(f"Conf >= 0.9 (alta confianza): {(primary['conf'] >= 0.9).sum():,} ({(primary['conf'] >= 0.9).mean():.1%})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Barras de distribución
ax = axes[0]
bars = ax.bar(CATS, dist.values, color=[COLORS[c] for c in CATS],
              width=0.55, edgecolor='white')
ax.bar_label(bars,
             labels=[f'{v:,}\n({pct[c]:.1f}%)' for v, c in zip(dist.values, CATS)],
             padding=4, fontsize=9)
ax.set_title('Distribución de etiquetas — corpus completo', fontweight='bold')
ax.set_ylabel('Textos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax.set_ylim(0, dist.max() * 1.2)
ax.tick_params(axis='x', rotation=10)

# Histograma de confianza
ax2 = axes[1]
for cat in CATS:
    ax2.hist(primary[primary['label'] == cat]['conf'],
             bins=16, alpha=0.65, label=cat, color=COLORS[cat],
             range=(0.2, 1.0), density=True)
ax2.set_title('Distribución de confianza por etiqueta', fontweight='bold')
ax2.set_xlabel('Confianza')
ax2.set_ylabel('Densidad')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()


## 3. Distribución por país

> Basado en **`crossval.csv`** (muestra estratificada de 1 500 textos con metadata completa).  
> Para el análisis sobre los 392 k textos primarios, ver §5 (requiere dataset original con `country`).


In [ ]:
country_label = (
    crossval.groupby(['country', 'majority'])
    .size().unstack(fill_value=0)
    .reindex(columns=CATS, fill_value=0)
)
country_label['n'] = country_label.sum(axis=1)
country_pct = country_label[CATS].div(country_label['n'], axis=0).mul(100).round(1)

display(
    country_pct
    .join(country_label['n'])
    .sort_values('PRO_EQUALITY', ascending=False)
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap %
order = country_pct.sort_values('PRO_EQUALITY', ascending=False).index
sns.heatmap(
    country_pct.loc[order],
    annot=True, fmt='.0f', cmap='Blues',
    linewidths=0.4, ax=axes[0],
    cbar_kws={'label': '%'}
)
axes[0].set_title('% por etiqueta y país (voto mayoritario)', fontweight='bold')
axes[0].set_xlabel('')

# Barras apiladas horizontales
ax2 = axes[1]
order2 = country_pct.sort_values('ANTI_EQUALITY').index
bottom = np.zeros(len(order2))
for cat in CATS:
    ax2.barh(order2, country_pct.loc[order2, cat],
             left=bottom, color=COLORS[cat], label=cat, height=0.7)
    bottom += country_pct.loc[order2, cat].values
ax2.set_xlabel('% de textos')
ax2.set_title('Composición por país (% apilado)', fontweight='bold')
ax2.legend(bbox_to_anchor=(1.01, 1), fontsize=8)
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.show()


## 4. Distribución por dimensión

In [ ]:
# Crossval
dim_cv = (
    crossval.groupby(['dimension', 'majority'])
    .size().unstack(fill_value=0)
    .reindex(columns=CATS, fill_value=0)
)
dim_cv['n'] = dim_cv.sum(axis=1)
dim_cv_pct = dim_cv[CATS].div(dim_cv['n'], axis=0).mul(100).round(1)

# Manual (etiqueta humana)
man_clean = manual.dropna(subset=['label_primary', 'label_manual'])
man_clean = man_clean[man_clean['label_manual'].isin(CATS)]

dim_man = (
    man_clean.groupby(['dimension', 'label_manual'])
    .size().unstack(fill_value=0)
    .reindex(columns=CATS, fill_value=0)
)
dim_man['n'] = dim_man.sum(axis=1)
dim_man_pct = dim_man[CATS].div(dim_man['n'], axis=0).mul(100).round(1)

print('Crossval — voto mayoritario por dimensión (%)')
display(dim_cv_pct.join(dim_cv['n']))
print('\nManual — etiqueta humana por dimensión (%)')
display(dim_man_pct.join(dim_man['n']))


In [ ]:
x = np.arange(len(DIMS))
w = 0.26
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)

for ax, data_pct, title in [
    (axes[0], dim_cv_pct,  'Crossval (voto mayoritario)'),
    (axes[1], dim_man_pct, 'Manual labeling'),
]:
    for i, cat in enumerate(CATS):
        ax.bar(x + (i - 1) * w,
               data_pct.reindex(DIMS)[cat].values,
               width=w, color=COLORS[cat], label=cat, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(DIMS, rotation=15)
    ax.set_ylabel('% textos')
    ax.set_ylim(0, 85)
    ax.set_title(f'Distribución por dimensión — {title}', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 5. Análisis por país-mes (requiere dataset original)

> `primary.csv` sólo tiene `id, label, conf`. Para análisis temporal hay que hacer  
> join con `AllDimensions.csv` (el dataset original del pipeline).
> Si tienes Google Drive montado y la ruta es correcta, esta celda se ejecuta sola.


In [ ]:
if os.path.exists(ORIG_PATH):
    # Cargar sólo columnas necesarias (adaptar si la columna de fecha tiene otro nombre)
    usecols = lambda c: c in {'id', 'country', 'dimension', 'date', 'month', 'year', 'published_date'}
    orig = pd.read_csv(ORIG_PATH, usecols=usecols)
    orig['id'] = orig['id'].astype(str)
    primary['id'] = primary['id'].astype(str)

    print('Columnas disponibles en AllDimensions.csv:', orig.columns.tolist())

    # Detectar columna de fecha
    date_col = next((c for c in ['date', 'published_date', 'month', 'year'] if c in orig.columns), None)
    if date_col in ('date', 'published_date'):
        orig['year_month'] = pd.to_datetime(orig[date_col], errors='coerce').dt.to_period('M').astype(str)
    elif date_col:
        orig['year_month'] = orig[date_col].astype(str)
    else:
        print('No se encontró columna de fecha. Revisa los nombres de columna arriba.')
        date_col = None

    merge_cols = ['id', 'country', 'dimension'] + (['year_month'] if date_col else [])
    primary_full = primary.merge(orig[merge_cols], on='id', how='left')

    if date_col and 'year_month' in primary_full.columns:
        # ── Tendencia mensual global ─────────────────────────────────────
        trend = (
            primary_full.groupby(['year_month', 'label'])
            .size().unstack(fill_value=0)
            .reindex(columns=CATS, fill_value=0)
        )
        trend_pct = trend.div(trend.sum(axis=1), axis=0).mul(100)

        fig, ax = plt.subplots(figsize=(14, 4))
        for cat in CATS:
            ax.plot(trend_pct.index, trend_pct[cat],
                    marker='o', ms=3, color=COLORS[cat], label=cat)
        ax.set_title('Evolución mensual de etiquetas — corpus completo', fontweight='bold')
        ax.set_xlabel('Mes'); ax.set_ylabel('% textos')
        ax.legend(); plt.xticks(rotation=45, ha='right')
        plt.tight_layout(); plt.show()

        # ── Heatmap país × mes (n textos por etiqueta) ──────────────────
        for cat in CATS:
            pivot = (
                primary_full[primary_full['label'] == cat]
                .groupby(['country', 'year_month'])
                .size().unstack(fill_value=0)
            )
            if pivot.shape[1] < 3:
                continue
            fig, ax = plt.subplots(figsize=(16, 6))
            sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.2,
                        ax=ax, cbar_kws={'label': 'n textos'})
            ax.set_title(f'Textos {cat} por país y mes', fontweight='bold')
            plt.tight_layout(); plt.show()
else:
    print('Dataset original no encontrado en:')
    print(' ', ORIG_PATH)
    print('Monta Google Drive y verifica la ruta para ejecutar este análisis.')


## 6. Validación cruzada — acuerdo entre modelos

**Modelos:** Gemini Flash 2.0 Lite · Groq Llama-3.1-8B · OpenRouter Llama-3.3-70B  
**Acuerdo:** Cohen's κ (> 0.61 = sustancial · > 0.81 = casi perfecto)


In [ ]:
PAIRS = [
    ('label_gemini', 'label_groq', 'Gemini vs Groq'),
    ('label_gemini', 'label_or',   'Gemini vs OpenRouter'),
    ('label_groq',   'label_or',   'Groq vs OpenRouter'),
]

def kappa_table(df):
    rows = []
    for c1, c2, name in PAIRS:
        k = cohen_kappa_score(df[c1], df[c2])
        a = (df[c1] == df[c2]).mean()
        val = ('casi perfecto' if k >= 0.81 else
               'sustancial'   if k >= 0.61 else
               'moderado'     if k >= 0.41 else 'bajo')
        rows.append({'Par': name, 'κ': round(k, 3), '% acuerdo': f'{a:.1%}', 'Valoración': val})
    return pd.DataFrame(rows).set_index('Par')

print('=== ACUERDO GLOBAL ===')
display(kappa_table(crossval))


In [ ]:
print('=== ACUERDO POR DIMENSIÓN ===')
for dim in DIMS:
    sub = crossval[crossval['dimension'] == dim]
    print(f'\n{dim.upper()} (n={len(sub)})')
    display(kappa_table(sub))


In [ ]:
# Matrices de confusión entre pares
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Matrices de confusión entre pares de modelos (crossval)',
             fontweight='bold', fontsize=12)

for ax, (c1, c2, title) in zip(axes, PAIRS):
    cm = confusion_matrix(crossval[c1], crossval[c2],
                          labels=CATS, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=CATS, yticklabels=CATS,
                linewidths=0.5, annot_kws={'size': 8})
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Columna', fontsize=8)
    ax.set_ylabel('Fila', fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# Consenso y patrones de desacuerdo
print(f"Consenso completo (3/3) : {(crossval['consensus'] == 1.0).mean():.1%}")
print(f"Mayoría (2+/3)          : {(crossval['consensus'] >= 2/3).mean():.1%}")
print(f"Desacuerdo total        : {(crossval['consensus'] < 2/3).mean():.1%}")

discord = crossval[crossval['consensus'] < 1.0].copy()
discord['pattern'] = discord.apply(
    lambda r: ' | '.join(sorted([r['label_gemini'], r['label_groq'], r['label_or']])), axis=1)
print('\nPatrones de desacuerdo más frecuentes:')
display(discord['pattern'].value_counts().head(10).to_frame())


## 7. F1 por categoría y dimensión (crossval)

> **Referencia:** voto mayoritario (los 3 modelos).  
> F1 de cada modelo individual frente a esa referencia.


In [ ]:
# F1 por categoría — global
f1_rows = []
for model, col in MODELS.items():
    rep = classification_report(
        crossval['majority'], crossval[col],
        labels=CATS, output_dict=True, zero_division=0
    )
    for cat in CATS:
        f1_rows.append({
            'Modelo': model, 'Categoría': cat,
            'F1': round(rep[cat]['f1-score'], 3),
            'Precision': round(rep[cat]['precision'], 3),
            'Recall': round(rep[cat]['recall'], 3),
            'Support': int(rep[cat]['support']),
        })

f1_df = pd.DataFrame(f1_rows)
print('F1 por categoría (vs voto mayoritario)')
display(f1_df.pivot_table(index='Categoría', columns='Modelo', values='F1').round(3))
print('\nPrecision:')
display(f1_df.pivot_table(index='Categoría', columns='Modelo', values='Precision').round(3))
print('\nRecall:')
display(f1_df.pivot_table(index='Categoría', columns='Modelo', values='Recall').round(3))


In [ ]:
# F1 macro por modelo × dimensión — heatmap
f1_mac_rows = []
for dim in DIMS:
    sub = crossval[crossval['dimension'] == dim]
    for model, col in MODELS.items():
        fm = f1_score(sub['majority'], sub[col],
                      labels=CATS, average='macro', zero_division=0)
        f1_mac_rows.append({'Dimensión': dim, 'Modelo': model, 'F1 macro': round(fm, 3)})

heat = pd.DataFrame(f1_mac_rows).pivot(
    index='Dimensión', columns='Modelo', values='F1 macro')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='YlGn',
            vmin=0.5, vmax=1.0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'F1 macro'})
ax.set_title('F1 macro por modelo y dimensión (ref. = voto mayoritario)', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# F1 por categoría × dimensión para cada modelo
f1_dim_rows = []
for dim in DIMS:
    sub = crossval[crossval['dimension'] == dim]
    for model, col in MODELS.items():
        rep = classification_report(
            sub['majority'], sub[col],
            labels=CATS, output_dict=True, zero_division=0
        )
        for cat in CATS:
            f1_dim_rows.append({
                'Dimensión': dim, 'Modelo': model, 'Categoría': cat,
                'F1': round(rep[cat]['f1-score'], 3)
            })

f1_dim_df = pd.DataFrame(f1_dim_rows)
for model in MODELS:
    print(f'\n{model}')
    display(
        f1_dim_df[f1_dim_df['Modelo'] == model]
        .pivot(index='Dimensión', columns='Categoría', values='F1')
    )


## 8. Validación manual — Primary vs. Humano

750 textos etiquetados a mano (50 por categoría × dimensión).  
**Referencia:** `label_manual` (humano) · **Predicción:** `label_primary` (modelo).


In [ ]:
acc = (man_clean['label_primary'] == man_clean['label_manual']).mean()
print(f'Acuerdo global (accuracy): {acc:.1%}  '
      f'({int(acc * len(man_clean))}/{len(man_clean)} textos)')
print()
print('Reporte de clasificación completo:')
print(classification_report(
    man_clean['label_manual'], man_clean['label_primary'],
    labels=CATS, zero_division=0
))


In [ ]:
# Tabla F1 por categoría
rep = classification_report(
    man_clean['label_manual'], man_clean['label_primary'],
    labels=CATS, output_dict=True, zero_division=0
)
f1_cat = pd.DataFrame(
    {cat: {'Precision': round(rep[cat]['precision'], 3),
           'Recall':    round(rep[cat]['recall'], 3),
           'F1':        round(rep[cat]['f1-score'], 3),
           'Support':   int(rep[cat]['support'])} for cat in CATS}
).T
f1_cat.loc['macro avg'] = [
    round(rep['macro avg']['precision'], 3),
    round(rep['macro avg']['recall'], 3),
    round(rep['macro avg']['f1-score'], 3),
    int(rep['macro avg']['support'])
]
display(f1_cat)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(
    man_clean['label_manual'], man_clean['label_primary'],
    labels=CATS, normalize='true'
)
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=axes[0],
            xticklabels=CATS, yticklabels=CATS, linewidths=0.5)
axes[0].set_title('Matriz de confusión\nfila = manual (ref), col = primary (pred)',
                  fontweight='bold')
axes[0].set_xlabel('Predicción (primary)', fontsize=9)
axes[0].set_ylabel('Referencia (manual)', fontsize=9)
axes[0].tick_params(labelsize=8)

# F1 por dimensión (heatmap)
f1_dim_man = []
for dim in DIMS:
    sub = man_clean[man_clean['dimension'] == dim]
    if len(sub) == 0:
        continue
    rep_d = classification_report(
        sub['label_manual'], sub['label_primary'],
        labels=CATS, output_dict=True, zero_division=0
    )
    for cat in CATS:
        f1_dim_man.append({'Dimensión': dim, 'Categoría': cat,
                           'F1': round(rep_d[cat]['f1-score'], 3)})

pivot_man = pd.DataFrame(f1_dim_man).pivot(
    index='Dimensión', columns='Categoría', values='F1')
sns.heatmap(pivot_man, annot=True, fmt='.3f', cmap='YlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=axes[1],
            cbar_kws={'label': 'F1'})
axes[1].set_title('F1 por dimensión y categoría\n(primary vs manual)', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# F1 por país (manual)
f1_ctry_rows = []
for ctry in sorted(man_clean['country'].unique()):
    sub = man_clean[man_clean['country'] == ctry]
    if len(sub) < 5:
        continue
    fm  = f1_score(sub['label_manual'], sub['label_primary'],
                   labels=CATS, average='macro', zero_division=0)
    acc_c = (sub['label_primary'] == sub['label_manual']).mean()
    f1_ctry_rows.append({'País': ctry, 'F1 macro': round(fm, 3),
                         'Accuracy': round(acc_c, 3), 'n': len(sub)})

f1_ctry = (pd.DataFrame(f1_ctry_rows)
           .set_index('País')
           .sort_values('F1 macro', ascending=False))
display(f1_ctry)


In [ ]:
# Ejemplos de desacuerdo primary vs manual
desac = man_clean[
    man_clean['label_primary'] != man_clean['label_manual']
][['id', 'dimension', 'country', 'label_primary', 'label_manual', 'notes', 'text']]

n_desac = len(desac)
print(f'Total desacuerdos: {n_desac} ({n_desac / len(man_clean):.1%})')
print('\nDistribución de errores por par (primary → manual):')
desac['error'] = desac['label_primary'] + ' → ' + desac['label_manual']
display(desac['error'].value_counts().to_frame())
print('\nMuestra de 15 casos de desacuerdo:')
display(desac.drop(columns='error').head(15))


## 9. Resumen ejecutivo

In [ ]:
k_gg  = cohen_kappa_score(crossval['label_gemini'], crossval['label_groq'])
k_go  = cohen_kappa_score(crossval['label_gemini'], crossval['label_or'])
k_qo  = cohen_kappa_score(crossval['label_groq'],   crossval['label_or'])
f1_man = f1_score(man_clean['label_manual'], man_clean['label_primary'],
                  labels=CATS, average='macro', zero_division=0)

resumen = pd.DataFrame([
    {'Métrica': 'Textos clasificados (primary)',    'Valor': f'{len(primary):,}'},
    {'Métrica': '% PRO_EQUALITY',                   'Valor': f"{(primary['label']=='PRO_EQUALITY').mean():.1%}"},
    {'Métrica': '% ANTI_EQUALITY',                  'Valor': f"{(primary['label']=='ANTI_EQUALITY').mean():.1%}"},
    {'Métrica': '% NEUTRAL',                        'Valor': f"{(primary['label']=='NEUTRAL').mean():.1%}"},
    {'Métrica': 'Confianza media (primary)',         'Valor': f"{primary['conf'].mean():.3f}"},
    {'Métrica': '─────────────────────────────────', 'Valor': ''},
    {'Métrica': 'κ Gemini vs Groq',                 'Valor': f'{k_gg:.3f}'},
    {'Métrica': 'κ Gemini vs OpenRouter',           'Valor': f'{k_go:.3f}'},
    {'Métrica': 'κ Groq vs OpenRouter',             'Valor': f'{k_qo:.3f}'},
    {'Métrica': 'Consenso 3/3 en crossval',         'Valor': f"{(crossval['consensus']==1.0).mean():.1%}"},
    {'Métrica': '─────────────────────────────────', 'Valor': ''},
    {'Métrica': 'Accuracy primary vs manual',       'Valor': f'{acc:.1%}'},
    {'Métrica': 'F1 macro primary vs manual',       'Valor': f'{f1_man:.3f}'},
    {'Métrica': 'Textos en validación manual',      'Valor': f'{len(man_clean):,}'},
]).set_index('Métrica')

display(resumen)
